# NZZ ContextAI — Pipeline Runner

Dieses Notebook führt die komplette Pipeline aus:
1. **Referenzantworten generieren** — Gemma4 beantwortet jede Ground-Truth-Frage anhand des Original-Artikels
2. **Vollständige Evaluation** — Retrieval + Generation + MLflow-Logging

In [ ]:
import sys, importlib
sys.path.insert(0, "../src")
sys.path.insert(0, "../scripts")

import config, ingest, experiment, evaluate
importlib.reload(config)
importlib.reload(ingest)
importlib.reload(experiment)
importlib.reload(evaluate)

print("Imports OK")

## 1 — Indexierung

Lädt rohe JSON-Artikel, bereinigt sie (BeautifulSoup), chunked sie und speichert Embeddings in ChromaDB.
Nur nötig wenn neue Daten dazukommen oder die Collection zurückgesetzt werden soll.

- `month`: z.B. `'2025_12'` für einen einzelnen Monat, `None` für alle
- `reset=True`: löscht die bestehende Collection vor dem Indexieren

In [ ]:
ingest.ingest(month="2025_12", reset=True)

## 1 — Referenzantworten generieren

Ruft Gemma4 für jeden Ground-Truth-Eintrag ohne `expected_answer` auf und speichert die Antworten in `data/eval/ground_truth.jsonl`.
Einträge die bereits eine vollständige Antwort haben werden übersprungen.

In [2]:
import build_expected_answers
importlib.reload(build_expected_answers)

build_expected_answers.main()

Lade Artikel...
[1/14] Bereits vorhanden — übersprungen
[2/14] Bereits vorhanden — übersprungen
[3/14] Bereits vorhanden — übersprungen
[4/14] Bereits vorhanden — übersprungen
[5/14] Bereits vorhanden — übersprungen
[6/14] Bereits vorhanden — übersprungen
[7/14] Bereits vorhanden — übersprungen
[8/14] Bereits vorhanden — übersprungen
[9/14] Bereits vorhanden — übersprungen
[10/14] Bereits vorhanden — übersprungen
[11/14] Bereits vorhanden — übersprungen
[12/14] Bereits vorhanden — übersprungen
[13/14] Bereits vorhanden — übersprungen
[14/14] Bereits vorhanden — übersprungen

✓ 0 Referenzantworten generiert und in ground_truth.jsonl gespeichert.


## 2 — Ground Truth prüfen

Zeigt den aktuellen Stand der `ground_truth.jsonl` — wie viele Einträge eine Referenzantwort haben.

In [3]:
import json
from config import EVAL_GROUND_TRUTH

with open(EVAL_GROUND_TRUTH, encoding="utf-8") as f:
    samples = [json.loads(line) for line in f if line.strip()]

complete = [s for s in samples if len(s.get("expected_answer") or "") > 80]
print(f"Ground-Truth Einträge: {len(samples)}")
print(f"Mit Referenzantwort:   {len(complete)} / {len(samples)}")
print()
for s in samples:
    answer = s.get("expected_answer") or ""
    status = "✓" if len(answer) > 80 else "✗"
    print(f"  {status}  {s['query'][:65]}")

Ground-Truth Einträge: 14
Mit Referenzantwort:   14 / 14

  ✓  Was erwarten Schweizer Anlageprofis für die Börsenmärkte im Jahr 
  ✓  Werden in der Schweiz wieder Negativzinsen eingeführt?
  ✓  Wie verhält sich Putins Kriegsführung im Vergleich zu Stalin?
  ✓  Was zeigen sowjetische Archivdokumente über die Denkweise russisc
  ✓  Wie werden ukrainische Kinder in Russland umerzogen?
  ✓  Russland verschleppt ukrainische Kinder und gibt sie zur Adoption
  ✓  Ukrainischer Freiwilliger birgt tote russische Soldaten vom Schla
  ✓  Menschlichkeit im Krieg: Umgang mit gefallenen Feinden im Donbass
  ✓  Wie hat sich die Schweiz für die Zurückweisung jüdischer Flüchtli
  ✓  Kaspar Villiger Entschuldigung Bundesrat Holocaust 1995
  ✓  Warum kann man Grundstücke besitzen und wie entstand das Eigentum
  ✓  Geschichte des Landbesitzes Schweiz Immobilienpreise Eigentumsrec
  ✓  Wie beeinflusst künstliche Intelligenz das menschliche Denken und
  ✓  Menschliche Dummheit und Kognitionsforschung im Zeit

## 3 — Vollständige Pipeline + MLflow

Läuft Retrieval für alle Ground-Truth-Queries, generiert Antworten mit Gemma4, berechnet ROUGE-L gegen die Referenzantworten und loggt alles in MLflow.

In [4]:
from experiment import run_experiment

run_experiment()

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



═════════════════════════════════════════════════════════════════
  Experiment: rag-evaluation  |  Run: 2026-04-20_11-15
═════════════════════════════════════════════════════════════════
  Chunks in DB:   3,140
  Eval-Queries:   14
  Reranking:      an
─────────────────────────────────────────────────────────────────

  Query                                                 H@1  H@3  H@5    MRR  ROUGE
  ─────────────────────────────────────────────────────────────────────────────────
  Was erwarten Schweizer Anlageprofis für die Börsenmärkt    ✓    ✓    ✓  1.000  0.282
  Werden in der Schweiz wieder Negativzinsen eingeführt?    ✓    ✓    ✓  1.000  0.248
  Wie verhält sich Putins Kriegsführung im Vergleich zu S    ✓    ✓    ✓  1.000  0.235
  Was zeigen sowjetische Archivdokumente über die Denkwei    ✓    ✓    ✓  1.000  0.315
  Wie werden ukrainische Kinder in Russland umerzogen?    ✓    ✓    ✓  1.000  0.253
  Russland verschleppt ukrainische Kinder und gibt sie zu    ✓    ✓    ✓  1.000 

## 4 — MLflow UI starten

Nach dem Run die Ergebnisse im Browser anschauen:

In [5]:
import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, "-m", "mlflow", "ui",
     "--backend-store-uri", "sqlite:///../mlflow.db",
     "--port", "5000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print("MLflow UI läuft → http://localhost:5000")
print(f"(Prozess-ID: {proc.pid} — Kernel neu starten zum Beenden)")

MLflow UI läuft → http://localhost:5000
(Prozess-ID: 41051 — Kernel neu starten zum Beenden)


## 5 — Einzelne Query testen

In [ ]:
from embed import get_chroma_collection
from retrieval import load_models, retrieve
from generate import generate
from config import (
    CHROMA_PATH, CHROMA_COLLECTION,
    USE_RERANKING, EVAL_TOP_K_RETRIEVAL, EVAL_TOP_K_FINAL,
)

collection      = get_chroma_collection(CHROMA_PATH, CHROMA_COLLECTION)
model, reranker = load_models(use_reranking=USE_RERANKING)
print(f"Bereit — {collection.count():,} Chunks in ChromaDB")

In [ ]:
QUERY = "Was berichtete die NZZ über die SNB?"

chunks = retrieve(QUERY, model, collection, reranker,
                  top_k_retrieval=EVAL_TOP_K_RETRIEVAL,
                  top_k_rerank=EVAL_TOP_K_FINAL)

print(f"Top-{len(chunks)} Quellen:\n")
for i, c in enumerate(chunks, 1):
    score = c.get("rerank_score", c.get("similarity_score", 0))
    print(f"  [{i}] {c['title'][:65]}  (score: {score:.3f})")

print("\nAntwort:\n")
print(generate(QUERY, chunks))